In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv")

print(df.shape)
print("\nChurn split:")
print(df['Churn'].value_counts())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

(7043, 21)

Churn split:
Churn
No     5174
Yes    1869
Name: count, dtype: int64

Data types:
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

Missing values:
Series([], dtype: int64)


In [8]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
drop_cols = ['customerID']
df_clean = df.dropna()
df_clean.drop(columns=drop_cols, inplace=True)
sd = pd.get_dummies(df_clean.drop(columns=['Churn']), drop_first=True)
X = sd.values
y = df_clean['Churn'].map({'No': 0, 'Yes': 1}).values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Churn split:", y.sum(), "churned out of", len(y))

X shape: (7032, 30)
y shape: (7032,)
Churn split: 1869 churned out of 7032


In [9]:

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

pipeline_smote = Pipeline([
     ('scaler', StandardScaler()),
     ('model', LogisticRegression())
 ])
pipeline_smote.fit(X_train_sm, y_train_sm)
y_pred_sm = pipeline_smote.predict(X_test)
y_proba_sm = pipeline_smote.predict_proba(X_test)[:, 1]

print("\n=== SMOTE ===")
print(f"AUC: {roc_auc_score(y_test, y_proba_sm):.4f}")
print(classification_report(y_test, y_pred_sm, target_names=['No Churn', 'Churn']))


=== SMOTE ===
AUC: 0.8306
              precision    recall  f1-score   support

    No Churn       0.90      0.71      0.80      1033
       Churn       0.50      0.79      0.61       374

    accuracy                           0.73      1407
   macro avg       0.70      0.75      0.70      1407
weighted avg       0.79      0.73      0.75      1407



In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import GridSearchCV


param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__penalty': ['l1', 'l2'],
    'model__solver': ['liblinear'],
}

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000)),
])

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV AUC:", round(grid.best_score_, 4))

# Evaluate best model on test set
best_model = grid.best_estimator_
y_pred_best = best_model.predict(X_test)
y_proba_best = best_model.predict_proba(X_test)[:, 1]

print("\nTest AUC:", round(roc_auc_score(y_test, y_proba_best), 4))
print("Test Accuracy:", round(best_model.score(X_test, y_test), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best, target_names=['No Churn', 'Churn']))
